# Cumulative charge tracker

### Imports

In [1]:

import os 
import sys
sys.path.append('./..')
import logging

import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt

import dynaphos
from dynaphos import utils
from dynaphos import cortex_models
from dynaphos.cortex_models import get_visual_field_coordinates_from_cortex_full, Map
from dynaphos.image_processing import canny_processor, sobel_processor
from dynaphos.simulator import GaussianSimulator as PhospheneSimulator
from dynaphos.utils import get_data_kwargs, to_numpy

In [2]:
# Load the simulator configuration file
params = utils.load_params('../config/params.yaml')

# Disable temporal dynamics and thresholding for static image demonstration
params['temporal_dynamics']['trace_increase_rate'] = 0.0  # No trace increase
params['temporal_dynamics']['trace_decay_per_second'] = 0.9999999  # Almost no trace decay
params['temporal_dynamics']['activation_decay_per_second'] = 0.999999  # Very slow activation decay

params['thresholding']['rheobase'] = 0.0  # No baseline threshold increase

# Get phosphene coordinates and initialize simulator
n_phosphenes = 100
phosphene_coords = cortex_models.get_visual_field_coordinates_probabilistically(params, n_phosphenes)
simulator = PhospheneSimulator(params, phosphene_coords)

# Stop after a couple of seconds
framerate = params['run']['fps']
max_n_frames = 60*framerate

# Load the videostream
cap = cv2.VideoCapture('./gradient_static.mp4')
if not cap.isOpened():
    print('Unable to read file :(')

# Set the output stream
fourcc = cv2.VideoWriter_fourcc(*'XVID') #codec
out = cv2.VideoWriter('clip_phosphenes.avi', fourcc, framerate, (512,256),False)

# Loop over frames
# Loop over frames
frame_nr = 0
while frame_nr<max_n_frames:
    # load next frame
    ret, frame = cap.read()
    frame_nr+=1
    
    if not ret:
        break 
        
    # to one channel, grayscale 
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # get square if frame is not square
    if frame.shape[0] != frame.shape[1]:
        shortest_side = min(frame.shape)
        frame = frame[frame.shape[0]//2-shortest_side//2:frame.shape[0]//2+shortest_side//2,
                      frame.shape[1]//2-shortest_side//2:frame.shape[1]//2+shortest_side//2]
        
    # preprocess: ONLY resize and blur (no edge detection)
    frame = cv2.resize(frame, (256,256))
    frame = cv2.GaussianBlur(frame, (21,21), 5)

    # Use the blurred frame directly (no edge detection)
    processed_img = frame
    stim_pattern = simulator.sample_stimulus(processed_img, rescale=True)

    # Generate phosphenes 
    phs = simulator(stim_pattern).clamp(0,1)
    phs = to_numpy(phs)*255

    # Check charge status periodically (every 30 frames to match logging)
    if frame_nr % 30 == 0:
        charge_status = simulator.get_charge_status()
        print(f"Frame {frame_nr}: Electrode {charge_status['max_electrode_idx']} = {charge_status['max_charge_uC']:.1f} µC / {charge_status['limit_uC']} µC")

    # Concatenate results
    cat = np.concatenate([processed_img, phs], axis=1).astype('uint8')
    
    out.write(cat)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Final charge status
print("\n=== FINAL CHARGE STATUS ===")
charge_status = simulator.get_charge_status()
print(f"Electrode {charge_status['max_electrode_idx']} reached max: {charge_status['max_charge_uC']:.1f} µC")
print(f"Charge limit: {charge_status['limit_uC']} µC")

cap.release()
out.release() 
cv2.destroyAllWindows()

Frame 30: Electrode 94 = 3.9 µC / 30.0 µC
Frame 60: Electrode 94 = 7.8 µC / 30.0 µC
Frame 90: Electrode 94 = 11.7 µC / 30.0 µC
Frame 120: Electrode 94 = 15.6 µC / 30.0 µC
Frame 150: Electrode 94 = 19.5 µC / 30.0 µC
Frame 180: Electrode 94 = 23.5 µC / 30.0 µC
Frame 210: Electrode 94 = 27.4 µC / 30.0 µC


c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [94]. limit=30.0 µC, values=[30.095924377441406]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [94]. limit=30.0 µC, values=[30.22620964050293]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [94]. limit=30.0 µC, values=[30.356494903564453]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [94]. limit=30.0 µC, values=[30

Frame 240: Electrode 94 = 31.3 µC / 30.0 µC


c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [8, 39, 94]. limit=30.0 µC, values=[30.6600399017334, 31.080055236816406, 31.919918060302734]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [8, 39, 94]. limit=30.0 µC, values=[30.78518295288086, 31.206912994384766, 32.05020523071289]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [8, 39, 94]. limit=30.0 µC, values=[30.91032600402832, 31.333770751953125, 32.18049240112305]
  warnings.warn(msg)
c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./

Frame 270: Electrode 94 = 35.2 µC / 30.0 µC


...

c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]. limit=30.0 µC, values=[46.3192024230957, 118.73680114746094, 141.00369262695312, 174.8063201904297, 169.8639373779297, 124.5119400024414, 108.01982879638672, 87.40316009521484, 179.90103149414062, 116.26590728759766, 141.00369262695312, 59.36840057373047, 126.15735626220703, 142.64987182617188, 142.64987182617188, 139.35406494140625, 119.56230163574219, 47.00010681152344, 120.38634490966797, 145.12364196777344, 87.54598236083984, 121.211517333984

Frame 1470: Electrode 94 = 192.2 µC / 30.0 µC


c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]. limit=30.0 µC, values=[47.3432502746582, 121.36985778808594, 144.13064575195312, 178.6827850341797, 173.6310272216797, 127.2731704711914, 110.41533660888672, 89.34139251708984, 183.88735961914062, 118.84427642822266, 144.13064575195312, 60.68492889404297, 128.95504760742188, 145.81344604492188, 145.81344604492188, 142.44439697265625, 122.21366882324219, 48.04234313964844, 123.05602264404297, 148.34190368652344, 89.48421478271484, 123.899505615234

Frame 1500: Electrode 94 = 196.1 µC / 30.0 µC

=== FINAL CHARGE STATUS ===
Electrode 94 reached max: 196.1 µC
Charge limit: 30.0 µC


c:\Users\jorge\OneDrive - Radboud Universiteit\CNS\Year 2\Thesis\dynaphos\examples\./..\dynaphos\simulator.py:412: UserWarning: [ChargeGuard] Cumulative charge limit exceeded for electrodes [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 38, 39, 40, 41, 42, 43, 44, 45, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 71, 72, 74, 75, 76, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]. limit=30.0 µC, values=[48.11128616333008, 123.34465026855469, 146.47586059570312, 181.5901336669922, 176.4563446044922, 129.34408569335938, 112.21196746826172, 90.7950668334961, 186.87710571289062, 120.7780532836914, 146.47586059570312, 61.672325134277344, 131.05343627929688, 148.18612670898438, 148.18612670898438, 144.76214599609375, 124.20219421386719, 48.82402038574219, 125.05828094482422, 150.75559997558594, 90.9378890991211, 125.915496826171